# IRE Assignment 1 + 2 — Full Pipeline (Kaggle T4)

**Run ALL cells top-to-bottom. Never skip cells.**

---

## Cell Reference
| Cell | What it does | ~Time |
|------|-------------|-------|
| 1 | Check hardware | <1s |
| 2 | Clone/pull repo | <30s |
| 3 | Set env vars + paths | <1s |
| 4 | Install pip deps, uninstall JAX | ~2 min |
| 5 | Verify GPU FAISS | <1s |
| 6 | HuggingFace login (for MIND) | <1s |
| 7 | Download + extract MIND | ~10 min |
| 8 | Download + extract EB-NeRD | ~5 min |
| 9 | Build processed parquets (ETL) | ~5 min |
| 10 | Snapshot processed files | <1s |
| 11 | Compute MIND embeddings | ~20 min |
| 12 | Compute EB-NeRD embeddings | ~10 min |
| 13 | Snapshot embeddings | <1s |
| 14 | BM25 Recall@K - MIND | ~10 min |
| 15 | BM25 Recall@K - EB-NeRD | ~15 min |
| 16 | Semantic Recall@K - MIND | ~5 min |
| 17 | Semantic Recall@K - EB-NeRD | ~5 min |
| 18 | Snapshot retrieval results | <1s |
| 19 | LightGBM - MIND | ~20 min |
| 20 | LightGBM - EB-NeRD | ~20 min |
| 21 | Snapshot models | <1s |
| 22 | Evaluate LightGBM - MIND | ~2 min |
| 23 | Evaluate LightGBM - EB-NeRD | ~2 min |
| 24 | NRMS Baseline - MIND | ~30 min |
| 25 | NRMS Baseline - EB-NeRD | ~30 min |
| 26 | Ablation Study (4 variants) | ~20 min |
| 27 | Serving Benchmark | ~3 min |
| 28 | Package Codabench zips | <1s |
| 29 | Print ALL results | <1s |

In [ ]:
# CELL 1: Hardware Check
import os, subprocess, psutil, torch

print('=== DISK ===')
os.system('df -h /kaggle/working')

print('\n=== RAM ===')
vm = psutil.virtual_memory()
print(f'Total:     {vm.total/1e9:.1f} GB')
print(f'Available: {vm.available/1e9:.1f} GB')

print('\n=== GPU ===')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free_b, total_b = torch.cuda.mem_get_info()
    print(f'Name:  {props.name}')
    print(f'VRAM:  {props.total_memory/1e9:.1f} GB  |  Free: {free_b/1e9:.1f} GB')
else:
    print('NO GPU DETECTED')

In [ ]:
# CELL 2: Clone / Pull Repository
import os
REPO = '/kaggle/working/news-retrieval-system'
if not os.path.exists(REPO):
    !git clone https://github.com/imchaitanya0/news-retrieval-system.git
else:
    !cd /kaggle/working/news-retrieval-system && git pull origin main
print('Done')

In [ ]:
# CELL 3: Working Directory, Python Path, Environment Variables
import os, sys
%cd /kaggle/working/news-retrieval-system
sys.path.insert(0, '/kaggle/working/news-retrieval-system')

# Kaggle T4 has 4 CPU cores - use them all
os.environ['OMP_NUM_THREADS']  = '4'
os.environ['POLARS_MAX_THREADS'] = '4'
# CRITICAL: prevents PyTorch from hoarding fragmented VRAM blocks between batches
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

for d in [
    'data/raw/mind/train', 'data/raw/mind/val', 'data/raw/mind/test',
    'data/raw/ebnerd',
    'data/processed',
    'data/feature_store/embeddings/mind', 'data/feature_store/embeddings/ebnerd',
    'data/feature_store/bm25/mind',       'data/feature_store/bm25/ebnerd',
    'data/feature_store/semantic/mind',   'data/feature_store/semantic/ebnerd',
    'data/models', 'data/results', 'data/submissions',
    '/kaggle/working/snapshots',
]:
    os.makedirs(d, exist_ok=True)

print(f'Working dir: {os.getcwd()}')

In [ ]:
# CELL 4: Install Dependencies
# IMPORTANT: JAX must be removed FIRST - it grabs CUDA on import
# and causes bm25s to OOM during batch retrieval.
# faiss-gpu-cu12 is required for T4 (CUDA 12 build).

!pip uninstall -y -q jax jaxlib 2>/dev/null || true

!pip install -q \
    huggingface_hub bm25s rank_bm25 sentence-transformers \
    lightgbm polars tqdm scikit-learn psutil scipy

# Swap in GPU FAISS (remove any CPU build first)
!pip uninstall -y -q faiss-cpu faiss-gpu faiss-gpu-cu12 2>/dev/null || true
!pip install -q faiss-gpu-cu12

# Confirm JAX is gone
import importlib.util
print(f'JAX present after uninstall: {importlib.util.find_spec("jax") is not None}  (must be False)')

In [ ]:
# CELL 5: Verify GPU FAISS
# If this errors, restart kernel (Runtime -> Restart kernel) and re-run Cell 4
import faiss, numpy as np

print(f'FAISS version: {faiss.__version__}')
print(f'GPU support:   {hasattr(faiss, "GpuIndexFlatIP")}')

if not hasattr(faiss, 'GpuIndexFlatIP'):
    raise RuntimeError('GPU FAISS not available! Restart kernel and re-run Cell 4.')

# Quick functional test
res = faiss.StandardGpuResources()
cfg = faiss.GpuIndexFlatConfig(); cfg.device = 0
idx = faiss.GpuIndexFlatIP(res, 64, cfg)
test_emb = np.random.rand(50, 64).astype('float32')
idx.add(test_emb)
_, I = idx.search(test_emb[:1], 5)
assert I[0][0] == 0, 'Self-retrieval failed!'
print('GPU FAISS self-test passed.')

In [ ]:
# CELL 6: HuggingFace Login (needed for MIND download)
# Add your token: Kaggle sidebar -> Add-ons -> Secrets -> HF_TOKEN
# Get token from: https://huggingface.co/settings/tokens
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print('HuggingFace login successful')
except Exception as e:
    print(f'Login failed: {e}')
    print('Add your HF token via: Add-ons -> Secrets -> Name: HF_TOKEN')

In [ ]:
# CELL 7: Download + Extract MIND Large
# SKIP if data/raw/mind/train/behaviors.tsv already exists
import zipfile, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

if Path('data/raw/mind/train/behaviors.tsv').exists():
    print('MIND already extracted - skipping')
else:
    for fname, sub in [
        ('MINDlarge_train.zip', 'train'),
        ('MINDlarge_dev.zip',   'val'),
        ('MINDlarge_test.zip',  'test'),
    ]:
        local_path = hf_hub_download(
            repo_id='yjw1029/MIND', filename=fname,
            repo_type='dataset', local_dir='data/raw/mind',
        )
        extract_to = Path(f'data/raw/mind/{sub}')
        extract_to.mkdir(parents=True, exist_ok=True)
        # Find the downloaded file (HF may nest it)
        matches = list(Path('data/raw/mind').rglob(fname))
        zip_path = matches[0] if matches else Path(local_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        # Flatten any nested folder
        for tsv in ['behaviors.tsv', 'news.tsv']:
            hits = list(extract_to.rglob(tsv))
            if hits and hits[0].parent != extract_to:
                shutil.move(str(hits[0]), str(extract_to / tsv))
        zip_path.unlink(missing_ok=True)
        print(f'Extracted {fname} -> {sub}')

for split in ['train', 'val', 'test']:
    b = Path(f'data/raw/mind/{split}/behaviors.tsv').exists()
    n = Path(f'data/raw/mind/{split}/news.tsv').exists()
    print(f'  {split}: behaviors={b}, news={n}')

In [ ]:
# CELL 8: Download + Extract EB-NeRD Small
# SKIP if data/raw/ebnerd/train/behaviors.parquet exists
import requests, zipfile, shutil
from pathlib import Path
from tqdm import tqdm

if Path('data/raw/ebnerd/train/behaviors.parquet').exists():
    print('EB-NeRD already extracted - skipping')
else:
    for fname, url in [
        ('ebnerd_small.zip',   'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_small.zip'),
        ('ebnerd_testset.zip', 'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_testset.zip'),
    ]:
        if not Path(fname).exists():
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                total = int(r.headers.get('content-length', 0))
                with open(fname, 'wb') as f:
                    with tqdm(total=total, unit='B', unit_scale=True, desc=fname) as bar:
                        for chunk in r.iter_content(65536):
                            f.write(chunk); bar.update(len(chunk))
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('data/raw/ebnerd')
        Path(fname).unlink()
        print(f'Extracted {fname}')

    # EB-NeRD uses 'validation', our code expects 'val'
    root = Path('data/raw/ebnerd')
    if (root / 'validation').exists() and not (root / 'val').exists():
        shutil.copytree(root / 'validation', root / 'val')
        print('Copied validation/ -> val/')

for split in ['train', 'val', 'test']:
    b = Path(f'data/raw/ebnerd/{split}/behaviors.parquet').exists()
    print(f'  {split}/behaviors.parquet: {b}')

In [ ]:
# CELL 9: ETL Pipeline - Build Unified Processed Parquets
# Reads raw MIND TSVs and EB-NeRD parquets.
# Applies schema normalization:
#   - Safe rename (skips duplicate columns like 'category')
#   - subcategory List[Int16] -> comma-separated string
#   - Adds 'dataset' column ('mind' or 'ebnerd')
#
# Outputs (in data/processed/):
#   articles_mind.parquet         ~130,379 rows
#   articles_ebnerd.parquet       ~125,541 rows
#   behaviors_mind_train.parquet  ~2.2M rows
#   behaviors_mind_val.parquet    ~376K rows
#   behaviors_mind_test.parquet   ~2.37M rows
#   behaviors_ebnerd_*.parquet
from pathlib import Path
import polars as pl

if (Path('data/processed/articles_mind.parquet').exists() and
        Path('data/processed/articles_ebnerd.parquet').exists()):
    print('Processed parquets already exist - skipping ETL')
else:
    !python -m src.data.build_pipeline

print('\n=== Processed Parquets ===')
for f in sorted(Path('data/processed').glob('*.parquet')):
    n = pl.scan_parquet(f).select(pl.len()).collect().item()
    print(f'  {f.name:55s}  {f.stat().st_size/1e6:7.1f} MB  {n:>10,} rows')

In [ ]:
# CELL 10: Snapshot Processed Parquets
# Click 'Save Version' (top-right) after this cell!
import os, shutil
snap = '/kaggle/working/snapshots/processed'
os.makedirs(snap, exist_ok=True)
n = 0
for f in os.listdir('data/processed'):
    shutil.copy2(f'data/processed/{f}', f'{snap}/{f}')
    n += 1
print(f'{n} parquets snapshotted -> {snap}')
print('-> Click Save Version now!')

In [ ]:
# CELL 11: Compute Sentence-Transformer Embeddings - MIND
# Model: paraphrase-multilingual-MiniLM-L12-v2
# Output dim: 384 (L2-normalised float32)
# ~130,379 articles -> ~200 MB .npy file
# Cached after first run - safe to re-run.
# Output: data/feature_store/embeddings/mind/article_embeddings.npy
# Time: ~15-25 min on T4
import gc, torch, polars as pl
from src.retrieval.semantic import load_or_compute_embeddings

gc.collect(); torch.cuda.empty_cache()

articles_mind = pl.read_parquet('data/processed/articles_mind.parquet')
print(f'MIND articles: {len(articles_mind):,}')

embs_mind, ids_mind = load_or_compute_embeddings(articles_mind, 'mind')
print(f'Embeddings: shape={embs_mind.shape}, dtype={embs_mind.dtype}, size={embs_mind.nbytes/1e6:.1f} MB')

# Free large numpy array from RAM (it's safely on disk)
del articles_mind, embs_mind, ids_mind
gc.collect()

In [ ]:
# CELL 12: Compute Sentence-Transformer Embeddings - EB-NeRD
# ~125,541 articles -> ~192 MB .npy file
# Output: data/feature_store/embeddings/ebnerd/article_embeddings.npy
# Time: ~10-15 min on T4
import gc, torch, polars as pl
from src.retrieval.semantic import load_or_compute_embeddings

gc.collect(); torch.cuda.empty_cache()

articles_eb = pl.read_parquet('data/processed/articles_ebnerd.parquet')
print(f'EB-NeRD articles: {len(articles_eb):,}')

embs_eb, ids_eb = load_or_compute_embeddings(articles_eb, 'ebnerd')
print(f'Embeddings: shape={embs_eb.shape}, dtype={embs_eb.dtype}, size={embs_eb.nbytes/1e6:.1f} MB')

del articles_eb, embs_eb, ids_eb
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# CELL 13: Snapshot Embeddings - Click 'Save Version' after this!
import os, shutil
from pathlib import Path
for ds in ['mind', 'ebnerd']:
    src = f'data/feature_store/embeddings/{ds}'
    dst = f'/kaggle/working/snapshots/embeddings/{ds}'
    if os.path.isdir(src):
        if os.path.isdir(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        mb = sum(f.stat().st_size for f in Path(dst).rglob('*') if f.is_file()) / 1e6
        print(f'{ds} embeddings snapshotted ({mb:.0f} MB)')
    else:
        print(f'WARNING: {ds} embeddings not found - did Cell 11/12 finish?')
print('-> Click Save Version now!')

In [ ]:
# CELL 14: BM25 Recall@K - MIND
# BM25 Okapi (k1=1.5, b=0.75) over article title+subtitle
# User query = concat of last 5 clicked article titles
# Search: scipy.sparse matmul - bypasses JAX/numba in bm25s
# batch_size=256: caps RAM to ~133 MB per batch
# Output: data/results/bm25_mind_val.json
# Time: ~10-15 min
# *** COPY PRINTED RECALL@K NUMBERS ***
!rm -rf data/feature_store/bm25/mind
!python -m src.retrieval.bm25 --dataset mind --split val --k 50 100 200

In [ ]:
# CELL 15: BM25 Recall@K - EB-NeRD
# 244,647 queries x 125,541 docs, 490 batches of 500 each
# Output: data/results/bm25_ebnerd_val.json
# Time: ~12-20 min
# *** COPY PRINTED RECALL@K NUMBERS ***
!rm -rf data/feature_store/bm25/ebnerd
!python -m src.retrieval.bm25 --dataset ebnerd --split val --k 50 100 200

In [ ]:
# CELL 16: Semantic (ANN) Recall@K - MIND
# Model: paraphrase-multilingual-MiniLM-L12-v2 (384-dim)
# Index: GPU FlatIP (exact cosine similarity)
# User vector: mean of last 10 clicked article embeddings (384-dim)
# Search: ALL 376K queries batched into one FAISS call -> seconds!
# Output: data/results/semantic_mind_val.json
# Time: ~3-5 min
# *** COPY PRINTED RECALL@K NUMBERS ***
import gc, torch
gc.collect(); torch.cuda.empty_cache()

# Delete stale/corrupt FAISS index cache
!rm -rf data/feature_store/semantic/mind
!python -m src.retrieval.semantic --dataset mind --split val --k 50 100 200

In [ ]:
# CELL 17: Semantic (ANN) Recall@K - EB-NeRD
# Output: data/results/semantic_ebnerd_val.json
# Time: ~3-5 min
# *** COPY PRINTED RECALL@K NUMBERS ***
import gc, torch
gc.collect(); torch.cuda.empty_cache()

!rm -rf data/feature_store/semantic/ebnerd
!python -m src.retrieval.semantic --dataset ebnerd --split val --k 50 100 200

In [ ]:
# CELL 18: Snapshot Retrieval Results - Click 'Save Version' after this!
import os, shutil, json
from pathlib import Path

for ds in ['mind', 'ebnerd']:
    for store in ['bm25', 'semantic']:
        src = f'data/feature_store/{store}/{ds}'
        dst = f'/kaggle/working/snapshots/{store}/{ds}'
        if os.path.isdir(src):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.isdir(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst)

os.makedirs('/kaggle/working/snapshots/results', exist_ok=True)
for f in Path('data/results').glob('*.json'):
    shutil.copy2(f, f'/kaggle/working/snapshots/results/{f.name}')

print('Retrieval results snapshotted')
print('\n=== Current Results ===')
for f in sorted(Path('data/results').glob('*.json')):
    with open(f) as fp:
        print(f'\n{f.name}:')
        print(json.dumps(json.load(fp), indent=2))
print('\n-> Click Save Version now!')

In [ ]:
# CELL 19: LightGBM LambdaMART Ranker - MIND
# 8 features per (user, candidate) pair:
#   1. sem_score        recency-weighted cosine sim (384-dim user vec . cand emb)
#   2. lex_overlap      Jaccard token overlap (title+subtitle)
#   3. log_popularity   log(1 + click_count from train)
#   4. hist_len         min(len(history), 50)
#   5. cat_match        cand_category == user_top_category
#   6. inv_position     1 / editorial_position
#   7. freshness        exp(-hours_since_publish / 24)
#   8. recency_sem_score cosine sim using only last-3 clicked article vectors
#
# max-train-rows=200000 prevents CPU RAM OOM (Kaggle limit: 16GB)
# Output: data/models/lgbm_mind.pkl
#         data/submissions/mind_val_lgbm.txt
#         data/submissions/mind_test_lgbm.txt
# Time: ~20-30 min
# *** COPY FULL OUTPUT ***
import gc, torch
gc.collect(); torch.cuda.empty_cache()

!python -m src.ranking.train_lgbm --dataset mind --max-train-rows 200000

In [ ]:
# CELL 20: LightGBM LambdaMART Ranker - EB-NeRD
# Output: data/models/lgbm_ebnerd.pkl
#         data/submissions/ebnerd_val_lgbm.txt
#         data/submissions/ebnerd_test_lgbm.txt
# Time: ~20-30 min
# *** COPY FULL OUTPUT ***
import gc, torch
gc.collect(); torch.cuda.empty_cache()

!python -m src.ranking.train_lgbm --dataset ebnerd --max-train-rows 200000

In [ ]:
# CELL 21: Snapshot Models + Submissions - Click 'Save Version' after!
import os, shutil
from pathlib import Path

for snap_dir, src_dir in [
    ('/kaggle/working/snapshots/models',      'data/models'),
    ('/kaggle/working/snapshots/submissions', 'data/submissions'),
]:
    os.makedirs(snap_dir, exist_ok=True)
    files = list(Path(src_dir).glob('*'))
    for f in files:
        shutil.copy2(f, f'{snap_dir}/{f.name}')
    print(f'{len(files)} files -> {snap_dir}')
print('-> Click Save Version now!')

In [ ]:
# CELL 22: Full Evaluation - MIND
# Metrics: AUC, MRR, nDCG@5, nDCG@10
# Slices: cold users (hist<=5), warm users, head/tail articles
# 95% Bootstrap CI (1000 resamples)
# Output: data/results/eval_mind_val_lgbm.json
# Time: ~2-5 min
# *** COPY FULL OUTPUT ***
!python -m src.evaluation.evaluate --dataset mind --strategy lgbm --split val

In [ ]:
# CELL 23: Full Evaluation - EB-NeRD
# Output: data/results/eval_ebnerd_val_lgbm.json
# Time: ~2-5 min
# *** COPY FULL OUTPUT ***
!python -m src.evaluation.evaluate --dataset ebnerd --strategy lgbm --split val

In [ ]:
# CELL 24: NRMS Neural News Recommendation - MIND
# Architecture (Wu et al., EMNLP 2019):
#   News Encoder: Word Emb (300d) -> Linear -> 4-head Self-Attn -> 200-dim
#   User Encoder: Stack 50 news vecs -> 4-head Self-Attn -> 200-dim
#   Score: dot(user_vec_200d, candidate_vec_200d)
#   Loss: cross-entropy over (1 pos + 4 neg) per impression
#
# VRAM estimate: ~4 GB for batch=32, seq_len=30, history=50
# max-train-rows=50000 and max-val-rows=5000 keep us under T4 limit
# Output: data/models/nrms_mind.pt
#         data/results/nrms_mind_val.json
# Time: ~30-45 min (3 epochs)
# *** COPY EPOCH-BY-EPOCH OUTPUT ***
import gc, torch
gc.collect(); torch.cuda.empty_cache()

!python -m src.models.train_nrms \
    --dataset mind \
    --epochs 3 \
    --max-train-rows 50000 \
    --max-val-rows 5000

In [ ]:
# CELL 25: NRMS Neural News Recommendation - EB-NeRD
# Output: data/models/nrms_ebnerd.pt
#         data/results/nrms_ebnerd_val.json
# Time: ~30-45 min
# *** COPY EPOCH-BY-EPOCH OUTPUT ***
import gc, torch
gc.collect(); torch.cuda.empty_cache()

!python -m src.models.train_nrms \
    --dataset ebnerd \
    --epochs 3 \
    --max-train-rows 50000 \
    --max-val-rows 5000

In [ ]:
# CELL 26: Ablation Study (4 LightGBM Variants)
# Variant A: sem_score only (pure semantic baseline)
# Variant B: sem_score + lex_overlap (+ lexical)
# Variant C: original 6 features (sem, lex, pop, hist, cat, pos)
# Variant D: all 8 features (+ freshness, recency_sem_score) <- full model
# Each variant: AUC, MRR, nDCG@5, nDCG@10 with 95% bootstrap CI
# max-val-rows=10000 prevents OOM
# Time: ~20-30 min total (4 variants x 2 datasets)
# *** COPY FULL OUTPUT ***
import gc, torch
gc.collect(); torch.cuda.empty_cache()

!python -m src.ranking.ablation --dataset mind  --max-val-rows 10000

gc.collect(); torch.cuda.empty_cache()

!python -m src.ranking.ablation --dataset ebnerd --max-val-rows 10000

In [ ]:
# CELL 27: Serving Benchmark
# Measures for 500 random user requests:
#   - p50, p95, p99 latency
#   - max QPS at p99 SLA
#   - FAISS index memory footprint (MB)
#   - LightGBM model file size
#   - 10x scale analysis
# Output: data/results/serving_mind.json
#         data/results/serving_ebnerd.json
# Time: ~3-5 min
# *** COPY FULL OUTPUT ***
!python -m src.evaluation.serving_benchmark --dataset mind  --n-requests 500
!python -m src.evaluation.serving_benchmark --dataset ebnerd --n-requests 500

In [ ]:
# CELL 28: Package Codabench Submission Zips
# MIND:    mind_submission.zip   contains prediction.txt
# EB-NeRD: ebnerd_submission.zip contains predictions.txt
# Upload to respective Codabench competitions.
import os, zipfile, shutil
from pathlib import Path

os.chdir('/kaggle/working/news-retrieval-system')

def package_submission(src_txt, zip_name, inner_name):
    src = Path(src_txt)
    if not src.exists():
        print(f'MISSING: {src_txt} - did Cell 19/20 finish?')
        return
    tmp = Path(f'/kaggle/working/{inner_name}')
    shutil.copy(src, tmp)
    zip_path = f'/kaggle/working/{zip_name}'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(tmp, arcname=inner_name)
    tmp.unlink()
    size_mb = Path(zip_path).stat().st_size / 1e6
    print(f'{zip_name}  ({size_mb:.1f} MB)')
    os.system(f'unzip -l {zip_path}')

package_submission(
    'data/submissions/mind_test_lgbm.txt',  'mind_submission.zip',  'prediction.txt')
package_submission(
    'data/submissions/ebnerd_test_lgbm.txt', 'ebnerd_submission.zip', 'predictions.txt')

In [ ]:
# CELL 29: Print ALL Results
# *** PASTE THIS ENTIRE OUTPUT TO CHAT ***
import json, os
from pathlib import Path

SEP = '=' * 70
print(SEP)
print('FINAL RESULTS - PASTE ENTIRE OUTPUT TO CHAT')
print(SEP)

jsons = sorted(Path('data/results').glob('*.json'))
if not jsons:
    print('No result JSONs found - check that evaluation cells finished.')
else:
    for f in jsons:
        print(f'\n{"="*60}\n{f.name}\n{"="*60}')
        with open(f) as fp:
            print(json.dumps(json.load(fp), indent=2))

print(f'\n{SEP}\nSUBMISSION ZIPS\n{SEP}')
for f in Path('/kaggle/working').glob('*.zip'):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

print(f'\n{SEP}\nSNAPSHOT FILES\n{SEP}')
import subprocess
result = subprocess.run(
    ['find', '/kaggle/working/snapshots', '-type', 'f'],
    capture_output=True, text=True
)
print(result.stdout)